# 实验5-2 基于深度学习的机器视觉 - 路标识别

# 1.实验介绍
## 1.1 实验背景
无人驾驶日益成熟，路标检测是其中一个基础任务，需要达到非常高的准确率来为后续的决策做支撑。

## 1.2 实验要求
a）建立深度神经网络模型，并尽可能将其调到最佳状态。   
b）绘制深度神经网络模型图、绘制并分析学习曲线。  
c）用准确率等指标对模型进行评估。

## 1.3 实验环境
可以使用基于 Python 的 OpenCV 库进行图像相关处理，使用 Numpy 库进行相关数值运算，使用 pytorch 等框架建立深度学习模型等。

## 1.4 注意事项
这里只是提供了作业最后的答案，不要考虑去运行这个程序，除非你氪金上服务器，不然GPU或者CPU必定会爆炸，或者，可能过一个星期都跑不完（主要是训练部分，剩下的能跑）。数据如果需要的话问我来要


## 1.5 参考资料
OpenCV：https://opencv-python-tutroals.readthedocs.io/en/latest/py_tutorials/py_tutorials.html  
Numpy：https://www.numpy.org/  
PyTorch：https://pytorch.org/

# 2.实验内容
## 2.1 介绍数据集
GTSRB 是一个多类图像识别数据集。

+ 43类交通标注
+ 超过50000张图片


+ 物品都是放在白板上在日光/室内光源下拍摄的，压缩后的尺寸为 512 * 384

导入数据集成功后路径：  
data_path = "../gstrb_data"

In [ ]:
import torch.optim as optim
import pickle
import time
from torch.autograd import Variable
from tqdm import tqdm
import torchvision
import torch.nn.functional as F
import torch.nn as nn
from PIL import Image
from torchvision import datasets, transforms
import torch
import glob
import os
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

## 2.2 划分验证集

### 2.2.1 划分之前的准备
因为挂载过来的数据比较凌乱，我们只需要到训练集数据，这里我们新建数据集路径./data 将原文件的train数据拷贝过来

### 2.2.2 划分数据

每个图片名称前六位表示类别，中间6位数表示组数，一组有30张图片，这里将前三组划分为验证集

In [ ]:
def initialize_data(folder):
    # make validation_data by using images 00000*, 00001* and 00002* in each class
    train_folder = folder + '/train_images'
    val_folder = folder + '/val_images'
    if not os.path.isdir(val_folder):
        print(val_folder + ' not found, making a validation set')
        os.mkdir(val_folder)
        for dirs in os.listdir(train_folder):
            os.mkdir(val_folder + '/' + dirs)
            for f in os.listdir(train_folder + '/' + dirs):
                if f[6:11] == ('00000') or f[6:11] == ('00001') or f[6:11] == ('00002'):
                    # move file to validation folder
                    os.rename(train_folder + '/' + dirs + '/' + f,
                              val_folder + '/' + dirs + '/' + f)

In [ ]:
data_path = "../gstrb_data"
initialize_data(data_path)

现在随机展示其中的 6 张图片

In [ ]:
# 获取数据名称列表
img_list = glob.glob(os.path.join(data_path, '*/*/*.png'))

# 打印数据集总量
print("数据集总数量:", len(img_list))

# 从数据名称列表 img_list 中随机选取 6 个。
for i, img_path in enumerate(random.sample(img_list, 6)):
    # 读取图片
    img = cv2.imread(img_path)
    # 将图片从 BGR 模式转为 RGB 模式
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # 将窗口设置为 2 行 3 列 6个子图
    plt.subplot(2, 3, i + 1)
    # 展示图片
    plt.imshow(img)
    # 不显示坐标尺寸
    plt.axis('off')

## 2.3 数据增强

In [ ]:
# data augmentation for training and test time
# Resize all images to 32 * 32 and normalize them to mean = 0 and standard-deviation = 1 based on statistics collected from the training set

data_transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.3337, 0.3064, 0.3171), (0.2672, 0.2564, 0.2629))
])

# Resize, normalize and jitter image brightness
data_jitter_brightness = transforms.Compose([
    transforms.Resize((32, 32)),
    #     transforms.ColorJitter(brightness=-5),
    transforms.ColorJitter(brightness=5),
    transforms.ToTensor(),
    transforms.Normalize((0.3337, 0.3064, 0.3171), (0.2672, 0.2564, 0.2629))
])

应用数据增强

In [ ]:
def char_order2int_order(charoder):
    a = []
    for i in range(43):
        a.append(str(i))
    a.sort()
    kv = {}
    for i in range(43):
        kv[i] = int(a[i])
    return kv[charoder]


def processing_data(data_path, batch_size, use_gpu):
    """
    数据处理
    :param data_path: 数据集路径
    :return:  train_loader,val_loader:处理后的训练集,验证集loader
    """
    # 这里是把字典序转为按数字大小排列的序号 比如10这个文件夹的图片，原本通过datasets.ImageFolder获取到的label是2（按字典序排列 0，1，10，11，12，13...），现在转换为10

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.ConcatDataset([
            datasets.ImageFolder(data_path + '/train_images',
                                 transform=data_transforms, target_transform=char_order2int_order),
            datasets.ImageFolder(data_path + '/train_images',
                                 transform=data_jitter_brightness, target_transform=char_order2int_order),

        ]),
        batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=use_gpu)

    val_loader = torch.utils.data.DataLoader(
        datasets.ImageFolder(data_path + '/val_images',
                             transform=data_transforms, target_transform=char_order2int_order),
        batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=use_gpu)
    return train_loader, val_loader

### 对单张图片应用变换器

In [ ]:
transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])
img = Image.open("../gstrb_data/train_images/0/00000_00003_00000.png")
img = transforms(img)
plt.imshow(img.permute(1, 2, 0))

## 2.4 网络架构


In [ ]:
nclasses = 43  # GTSRB as 43 classes


class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        # CNN layers
        self.conv1 = nn.Conv2d(3, 100, kernel_size=5)
        self.conv2 = nn.Conv2d(100, 150, kernel_size=3)
        self.conv3 = nn.Conv2d(150, 250, kernel_size=3)
        self.fc1 = nn.Linear(250 * 2 * 2, 350)
        self.fc2 = nn.Linear(350, nclasses)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = F.max_pool2d(F.relu(self.conv3(x)), 2)
        x = x.view(-1, 250 * 2 * 2)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

## 2.5 训练网络

### 2.5.1 训练超参

In [ ]:
# 参数设置
class args:
    data = "../gstrb_data"
    batch_size = 64
    epochs = 20
    lr = 0.0001
    seed = 1
    log_interval = 10


torch.manual_seed(args.seed)


if torch.cuda.is_available():
    use_gpu = True
    print("Using GPU")
else:
    use_gpu = False
    print("Using CPU")

train_loader, val_loader = processing_data(args.data, args.batch_size, use_gpu)
model = Net()

# 优化器和学习率调整
optimizer = optim.Adam(filter(lambda p: p.requires_grad,
                       model.parameters()), lr=args.lr)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5, verbose=True)

### 2.5.2 训练函数和验证函数

In [ ]:
model = Net()

def train(epoch):
    model.train()
    correct = 0
    training_loss = 0
    for batch_idx, (data, target) in enumerate(tqdm(train_loader, desc='Loader')):
        data, target = Variable(data), Variable(target)
        if use_gpu:
            data = data.cuda()
            target = target.cuda()
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target, reduction="sum")
        loss.backward()
        optimizer.step()
        max_index = output.max(dim=1)[1]
        correct += (max_index == target).sum()
        training_loss += loss
    print('\nTraining set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        training_loss /
        len(train_loader.dataset), correct, len(train_loader.dataset),
        100. * correct / len(train_loader.dataset)))
    return training_loss / len(train_loader.dataset), 100. * correct / len(train_loader.dataset)


def validation():
    from torch.autograd import Variable
    model.eval()
    validation_loss = 0
    correct = 0
    for batch_idx, (data, target) in enumerate(tqdm(val_loader, desc='Loader')):
        with torch.no_grad():
            data, target = Variable(data), Variable(target)
            if use_gpu:
                data = data.cuda()
                target = target.cuda()
            output = model(data)
            # sum up batch loss
            validation_loss += F.nll_loss(output,
                                          target, reduction="sum").data.item()
            # get the index of the max log-probability
            pred = output.data.max(1, keepdim=True)[1]
            correct += pred.eq(target.data.view_as(pred)).cpu().sum()

#     scheduler.step(np.around(validation_loss, 2))
    print('\nValidation set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        validation_loss /
        len(val_loader.dataset), correct, len(val_loader.dataset),
        100. * correct / len(val_loader.dataset)))
    return validation_loss/len(val_loader.dataset), 100. * correct / len(val_loader.dataset)

### 2.5.3 模型训练过程和图形化

In [ ]:
res = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}
# 开始训练
start = time.time()
if use_gpu:
    model.cuda()

for epoch in range(1, args.epochs + 1):
    loss, acc = train(epoch)
    res["loss"].append(loss.cpu())
    res["accuracy"].append(acc.cpu())
    loss, acc = validation()
    res["val_loss"].append(loss)
    res["val_accuracy"].append(acc)
    model_file = 'results/model_' + str(epoch) + '.pth'
    torch.save(model.state_dict(), model_file,
               _use_new_zipfile_serialization=False)
    print('\nSaved model to ' + model_file)

print("模型训练总时长：", time.time()-start)
with open("./results/res", "wb") as f:
    pickle.dump(res, f)

In [ ]:
def plot_training_history(res):
    """
    绘制模型的训练结果
    :param res: 模型的训练结果
    :return:
    """
    # 绘制模型训练过程的损失和平均损失
    # 绘制模型训练过程的损失值曲线，标签是 loss
    plt.plot(res['loss'], label='loss')

    # 绘制模型训练过程中的平均损失曲线，标签是 val_loss
    plt.plot(res['val_loss'], label='val_loss')

    # 绘制图例,展示出每个数据对应的图像名称和图例的放置位置
    plt.legend(loc='upper right')

    # 展示图片
    plt.show()

    # 绘制模型训练过程中的的准确率和平均准确率
    # 绘制模型训练过程中的准确率曲线，标签是 acc
    plt.plot(res['accuracy'], label='accuracy')

    # 绘制模型训练过程中的平均准确率曲线，标签是 val_acc
    plt.plot(res['val_accuracy'], label='val_accuracy')

    # 绘制图例,展示出每个数据对应的图像名称，图例的放置位置为默认值。
    plt.legend()

    # 展示图片
    plt.show()

In [ ]:
# 绘制模型训练过程曲线
# 加载gpu训练的结果
with open("./results/res", "rb") as f:
    res = pickle.load(f)
plot_training_history(res)

## 2.6 加载模型和模型评估

In [ ]:
_, val_loader = processing_data(args.data, args.batch_size, use_gpu=False)
model_file = './results/model_%d.pth' % (args.epochs-1)
model = Net()
model.load_state_dict(torch.load(model_file))

# model.eval()
validation()

## 2.7 作业
通过对以上步骤流程的了解，相信大家对深度学习有了深刻的认识，但是模型比较简单，准确率也不高，大家可以试着写自己的深度学习模型，并将其调到最佳状态。在训练模型等过程中如果需要**保存数据、模型**等请写到 **results** 文件夹。

### 2.7.1 训练深度学习模型

深度学习模型训练流程, 包含数据处理、创建模型、训练模型、模型保存、评价模型等。  
如果对训练出来的模型不满意, 你可以通过调整模型的参数等方法重新训练模型, 直至训练出你满意的模型。  
如果你对自己训练出来的模型非常满意, 则可以提交作业报告!  

注意：

1. 你可以在我们准好的接口中实现深度学习模型（若使用可以修改函数接口），也可以自己实现深度学习模型。
2. 写好代码后可以在 Py 文件中使用 GPU 进行模型训练

In [ ]:
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.ConcatDataset([
        datasets.ImageFolder(data_path + '/train_images',
                             transform=data_transforms, target_transform=char_order2int_order),
        datasets.ImageFolder(data_path + '/train_images',
                             transform=data_jitter_brightness, target_transform=char_order2int_order),

    ]),
    batch_size=args.batch_size, shuffle=True, num_workers=4, pin_memory=use_gpu)

val_loader = torch.utils.data.DataLoader(
    datasets.ImageFolder(data_path + '/val_images',
                         transform=data_transforms, target_transform=char_order2int_order),
    batch_size=args.batch_size, shuffle=False, num_workers=4, pin_memory=use_gpu)

for epoch in tqdm(range(args.epochs), desc="Epoch"):
    loss, acc = train(epoch)
    res["loss"].append(loss.cpu())
    res["accuracy"].append(acc.cpu())
    loss, acc = validation()
    res["val_loss"].append(loss)
    res["val_accuracy"].append(acc)
    model_file = 'results/model_' + str(epoch) + '.pth'
    torch.save(model.state_dict(), model_file,
               _use_new_zipfile_serialization=False)
    print('\nSaved model to ' + model_file)

### 2.7.2 模型预测

在下方的代码块中编写模型预测部分的代码，请勿在别的位置作答

In [ ]:
# 加载模型
my_model_path = 'results/model_99.pth'
my_model = torch.load(my_model_path)
model = Net()
model.load_state_dict(my_model)
model.eval()

# 预测
def predict(img):
    """
    加载模型和模型预测
    主要步骤:
        1.图片处理
        2.用加载的模型预测图片的类别
    :param img: PIL.Image 对象
    :return: int, 模型识别图片的类别
    """
    img = Image.open(img)
    img = data_transforms(img).unsqueeze(0)
    # 使用加载的模型进行预测
    y_predict = model(img).argmax(dim=1).item()
    return y_predict

In [ ]:
# 输入图片路径和名称
img_path = 'test_with_label_3.png'

# 打印该张图片的类别
print(predict(img_path))